# Notebook 07: Court Case Validation

**Pipeline layer H'.** Tests whether the computational framework agrees with judges on which narrative elements and combinations are stock under copyright doctrine. The highest-priority validation of the project.

## Purpose

Build a curated corpus of published copyright opinions that apply **scènes à faire** doctrine or the closely-related **merger doctrine**, extract from each opinion the specific narrative elements and combinations the judge identified as stock or unprotectable, then test whether the computational pipeline's rankings would have flagged the same elements and combinations.

Both doctrines exclude creative material from copyright. Scènes à faire excludes elements required by the genre (a sheriff in a Western, a haunted house in Horror). Merger doctrine excludes expressions forced by the idea (rules of a card game, factual recitations). Different mechanisms, same effect: judicially-identified material that cannot be protected because using it was effectively required. Both enter the validation headline.

## Run protocol

This notebook has six phases that each save a checkpoint. Re-running picks up from the latest checkpoint, so the notebook is **idempotent and resumable**. Phases C and D include manual-review steps where you (Mira) edit a CSV file before the notebook proceeds.

| Phase | What it does | Time | Manual step |
|---|---|---|---|
| A | Pull CourtListener search results for both queries | ~5 min | none |
| B | Fetch full opinion text per cluster | ~10 min | none |
| C | Extract passages + auto-triage cases by scope | ~5 min | **Yes, review case_triage CSV** |
| D | LLM extraction of judicially-identified elements | ~5 min | **Yes, review element extraction CSV** |
| E | Match judicial elements/combinations against pipeline | ~2 min | none |
| F | Compute P@K curves, Spearman ρ, build report | ~1 min | none |

Total automated time: under 30 minutes. Manual review (Phases C and D combined): budget half a day for careful work. The manual review is the load-bearing human-judgment step the validation rests on.

## Inputs you need to prepare before running

1. **CourtListener API token.** Sign in at https://www.courtlistener.com/sign-in/, find the token under your profile, store it as a Colab Secret named `COURTLISTENER_TOKEN`.
2. **Anthropic API key.** Already configured in Colab Secrets as `ANTHROPIC_API_KEY` (used by 02b and 03b).
3. **Pipeline outputs from earlier notebooks.** Three parquets must be on Drive at the paths shown in Cell 1. They are: `stockiness_long_classified.parquet`, `combination_stockiness.parquet`, `element_centrality_by_genre.parquet`. David is providing these as part of the hand-off package.

## Outputs

Everything writes to `v4_rebuild/07_court_cases/`. The headline outputs for David are `validation_report.txt`, `validation_table.parquet`, and the per-case match parquets. See the spec document `docs/Notebook_07_Specification.md` §4 for the full output schema.

In [ ]:
# ----------------------------------------------------------------------
# Setup and configuration
# ----------------------------------------------------------------------
!pip install -q anthropic rapidfuzz

import os, re, json, time, hashlib, pickle
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm

# --- Google Drive -----------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ------------------------------------------------------------
# Drive layout mirrors the rest of the pipeline. If you set up the project
# in a different Drive location, change PROJECT_ROOT to match.
PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                    'DGK Lab - Scene A Faire Computational')
V4 = PROJECT_ROOT / 'v4_rebuild'

# Inputs from the earlier pipeline
PATH_ENRICHED        = V4 / '03b_classification' / 'stockiness_long_classified.parquet'
PATH_COMBINATIONS    = V4 / '03c_network'        / 'combination_stockiness.parquet'
PATH_CENTRALITY      = V4 / '03c_network'        / 'element_centrality_by_genre.parquet'

# Output directory for this notebook
OUT_DIR = V4 / '07_court_cases'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_OPINIONS   = OUT_DIR / 'cache_opinions'   ; CACHE_OPINIONS.mkdir(exist_ok=True)
CACHE_EXTRACTION = OUT_DIR / 'cache_extraction' ; CACHE_EXTRACTION.mkdir(exist_ok=True)

PATH_RAW_SEARCH   = OUT_DIR / 'courtlistener_raw.parquet'
PATH_OPINION_TEXT = OUT_DIR / 'opinion_texts.parquet'
PATH_PASSAGES     = OUT_DIR / 'case_passages.parquet'
PATH_TRIAGE_AUTO  = OUT_DIR / 'case_triage_auto.parquet'
PATH_TRIAGE_CURATED  = OUT_DIR / 'case_triage_CURATED.parquet'
PATH_EXTRACT_AUTO    = OUT_DIR / 'judicial_elements_auto.parquet'
PATH_EXTRACT_CURATED = OUT_DIR / 'judicial_elements_CURATED.parquet'
PATH_ELEM_MATCHES    = OUT_DIR / 'element_matches.parquet'
PATH_COMB_MATCHES    = OUT_DIR / 'combination_matches.parquet'
PATH_VALIDATION_TBL  = OUT_DIR / 'validation_table.parquet'
PATH_REPORT          = OUT_DIR / 'validation_report.txt'

# --- API keys (from Colab Secrets — never hardcode) -------------------
from google.colab import userdata
try:
    COURTLISTENER_TOKEN = userdata.get('COURTLISTENER_TOKEN')
    assert COURTLISTENER_TOKEN, 'empty'
except Exception:
    raise RuntimeError(
        'COURTLISTENER_TOKEN not found in Colab Secrets. Open the key icon '
        'in the left sidebar, add a secret named COURTLISTENER_TOKEN with '
        'the token David provided, and enable notebook access. Do not '
        'paste the token into a cell.'
    )
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    assert ANTHROPIC_API_KEY, 'empty'
except Exception:
    raise RuntimeError(
        'ANTHROPIC_API_KEY not found in Colab Secrets. Add it as a Colab '
        'Secret named ANTHROPIC_API_KEY.'
    )

import anthropic
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# --- Run configuration -------------------------------------------------
COURTLISTENER_BASE   = 'https://www.courtlistener.com/api/rest/v4/'
COURTLISTENER_HEADERS = {'Authorization': f'Token {COURTLISTENER_TOKEN}'}
REQUEST_SLEEP        = 0.5     # seconds between API calls — be polite
REQUEST_RETRIES      = 5
REQUEST_TIMEOUT      = 30

MODEL_NAME           = 'claude-sonnet-4-6'
EXTRACTION_MAX_TOKENS = 2000
EXTRACTION_MAX_WORKERS = 4

# Genre list, identical to the rest of the pipeline. Hardcoded so the
# notebook is self-contained.
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western',
]

# Thresholds for "flagged" predictions (see spec §6)
HIGH_STOCKINESS_PERCENTILE = 0.80   # element-level
COMBINATION_TOP_HEADLINE = 0.95     # combination top-5% headline
COMBINATION_TOP_STRICT   = 0.99     # combination top-1% sensitivity
COMBINATION_TOP_GENEROUS = 0.90     # combination top-10% sensitivity

# K values for the precision/recall curve (spec §3 Phase F)
K_VALUES = [10, 25, 50, 100]

print('Setup complete.')
print(f'  Project root: {PROJECT_ROOT}')
print(f'  Output dir  : {OUT_DIR}')
print(f'  Model       : {MODEL_NAME}')
print(f'  Genres      : {len(CANONICAL_GENRES)}')

In [ ]:
# ----------------------------------------------------------------------
# Load pipeline outputs from the earlier notebooks (03b and 03c).
#
# These three parquets are the ground truth the validation tests against.
# If any of these paths fail, confirm with David that the v4_rebuild/
# folder structure is in place at the PROJECT_ROOT defined above.
# ----------------------------------------------------------------------

for path, label in [
    (PATH_ENRICHED,     'stockiness_long_classified.parquet'),
    (PATH_COMBINATIONS, 'combination_stockiness.parquet'),
    (PATH_CENTRALITY,   'element_centrality_by_genre.parquet'),
]:
    assert path.exists(), f'Missing pipeline output: {path}'
    print(f'  Found: {label}  ({path.stat().st_size/1e6:.1f} MB)')

print('\nLoading...')
stk_cls = pd.read_parquet(PATH_ENRICHED)
print(f'  stockiness_long_classified : {len(stk_cls):,} rows × {stk_cls.shape[1]} cols')

comb_stock = pd.read_parquet(PATH_COMBINATIONS)
print(f'  combination_stockiness     : {len(comb_stock):,} rows × {comb_stock.shape[1]} cols')

centrality = pd.read_parquet(PATH_CENTRALITY)
print(f'  element_centrality_by_genre: {len(centrality):,} rows × {centrality.shape[1]} cols')

# Precompute per-genre rank columns for the element-level matching.
# Rank is per (genre, element_type) sorted by ensemble descending; this
# gives the rank we report in the validation table. Rank=1 is the top
# element in the genre for its type.
stk_cls['rank_in_genre_by_type'] = (
    stk_cls.groupby(['genre','element_type'])['ensemble']
           .rank(method='dense', ascending=False)
           .astype(int)
)
# Percentile = (max_rank - rank + 1) / max_rank within the (genre, type).
# Top of the list is 1.0; bottom approaches 0.0.
group_max = stk_cls.groupby(['genre','element_type'])['rank_in_genre_by_type'].transform('max')
stk_cls['percentile_in_genre_by_type'] = (group_max - stk_cls['rank_in_genre_by_type'] + 1) / group_max

# Precompute per-genre rank for combination_stock as well.
comb_stock['combination_stock_rank_in_genre'] = (
    comb_stock.groupby('genre')['combination_stock']
              .rank(method='dense', ascending=False)
              .astype(int)
)
gmax = comb_stock.groupby('genre')['combination_stock_rank_in_genre'].transform('max')
comb_stock['combination_stock_percentile'] = (gmax - comb_stock['combination_stock_rank_in_genre'] + 1) / gmax

print('\nPipeline ranks precomputed.')
print(f'  per-genre noun rank=1 examples: '
      f'{stk_cls[(stk_cls["element_type"]=="noun") & (stk_cls.rank_in_genre_by_type==1)].head(3)["element"].tolist()}')

In [ ]:
# ----------------------------------------------------------------------
# Phase A — Corpus pull from CourtListener
#
# Two queries, unioned and deduplicated by cluster_id:
#   (1) "scènes à faire" OR "scenes a faire"
#   (2) "merger doctrine" copyright
#
# CHECKPOINT: writes to PATH_RAW_SEARCH. Re-running this cell loads the
# checkpoint instead of re-querying.
# ----------------------------------------------------------------------

class DailyCapReached(RuntimeError):
    """Raised when CourtListener returns 429 with a Retry-After longer than
    a few minutes — i.e. the account\'s daily quota is exhausted. The
    notebook\'s checkpoint preserves progress, so the right action is to
    halt cleanly and re-run after the cooldown rather than burning cycles
    in a retry loop."""
    def __init__(self, retry_after_seconds):
        hours = retry_after_seconds / 3600
        super().__init__(
            f'CourtListener daily-cap reached. Retry-After header reports '
            f'{retry_after_seconds:,} seconds (~{hours:.1f} hours).\n'
            f'\n'
            f'Your account\'s daily API quota has been spent. The per-cluster\n'
            f'cache in cache_opinions/ has preserved everything fetched so\n'
            f'far; re-running this cell after the cooldown will resume from\n'
            f'the next un-cached cluster automatically.\n'
            f'\n'
            f'To increase the daily quota, check the FLP membership tier on\n'
            f'CourtListener (https://free.law/membership) or email\n'
            f'info@free.law to request elevated researcher access.'
        )


# Threshold above which a Retry-After is treated as a daily-cap signal
# rather than a transient throttle. Ten minutes is well above any
# rolling-window throttle and well below any conceivable daily reset.
DAILY_CAP_RETRY_THRESHOLD = 600  # seconds


def courtlistener_get(url, params=None):
    """GET with retries and polite backoff. Distinguishes transient
    rate-limit responses (short Retry-After) from daily-cap responses
    (Retry-After of many hours), halting cleanly on the latter so the
    caller doesn\'t burn iterations in a useless retry loop."""
    last_err = None
    delay = 2.0
    for attempt in range(REQUEST_RETRIES):
        try:
            r = requests.get(url,
                             params=params,
                             headers=COURTLISTENER_HEADERS,
                             timeout=REQUEST_TIMEOUT)
            if r.status_code == 429:
                retry_after_str = r.headers.get('Retry-After', '60')
                try:
                    retry_after = int(retry_after_str)
                except ValueError:
                    retry_after = 60
                if retry_after > DAILY_CAP_RETRY_THRESHOLD:
                    raise DailyCapReached(retry_after)
                time.sleep(retry_after)
                continue
            if r.status_code in (502, 503, 504):
                # Transient server-side issue — wait and retry
                time.sleep(60)
                continue
            r.raise_for_status()
            return r.json()
        except DailyCapReached:
            raise  # do not retry; propagate up so the caller halts
        except requests.RequestException as e:
            last_err = e
            time.sleep(delay)
            delay = min(delay * 2, 60)
    raise RuntimeError(f'CourtListener request failed after {REQUEST_RETRIES} attempts: {last_err}')


def pull_all_for_query(query_string, query_label):
    """Cursor-paginated full pull for one query string. Tags each result
    with which query surfaced it so multi-query overlap is visible."""
    results = []
    url = COURTLISTENER_BASE + 'search/'
    params = {'q': query_string, 'type': 'o', 'order_by': 'score desc'}
    first = True
    page = 0
    with tqdm(desc=f'pull {query_label}', unit='page') as pbar:
        while url:
            data = courtlistener_get(url, params=params if first else None)
            page_results = data.get('results', [])
            for r in page_results:
                r['source_query'] = query_label
            results.extend(page_results)
            url = data.get('next')
            first = False
            page += 1
            pbar.update(1)
            time.sleep(REQUEST_SLEEP)
    return results


if PATH_RAW_SEARCH.exists():
    print(f'Loading existing Phase A checkpoint: {PATH_RAW_SEARCH}')
    raw_search = pd.read_parquet(PATH_RAW_SEARCH)
else:
    print('Running Phase A: pulling CourtListener search results...')
    q1 = pull_all_for_query(
        '"scènes à faire" OR "scenes a faire"',
        'scenes_a_faire',
    )
    q2 = pull_all_for_query(
        '"merger doctrine" copyright',
        'merger_doctrine',
    )
    raw = q1 + q2
    raw_df = pd.DataFrame(raw)

    # Some result keys are deeply nested; we only need a handful at this
    # stage. Pick the columns that exist and coerce types.
    keep_cols = [c for c in [
        'cluster_id', 'caseName', 'caseNameShort', 'dateFiled',
        'court', 'court_id', 'docketNumber', 'citeCount',
        'absolute_url', 'snippet', 'status', 'source_query'
    ] if c in raw_df.columns]
    raw_search = raw_df[keep_cols].copy()
    # cluster_id must be int for joins downstream
    raw_search['cluster_id'] = pd.to_numeric(raw_search['cluster_id'], errors='coerce').astype('Int64')
    raw_search = raw_search.dropna(subset=['cluster_id']).reset_index(drop=True)

    raw_search.to_parquet(PATH_RAW_SEARCH, index=False)
    print(f'Saved: {PATH_RAW_SEARCH}')

# Deduplicate by cluster_id (a case may surface from both queries)
unique_cases = raw_search.drop_duplicates(subset=['cluster_id']).reset_index(drop=True)
print(f'\nPhase A complete.')
print(f'  total result rows (with duplicates across queries): {len(raw_search):,}')
print(f'  unique cases by cluster_id                         : {len(unique_cases):,}')
print(f'  by source query:')
print(raw_search['source_query'].value_counts().to_string())
print(f'\nTop 5 courts:')
print(raw_search['court'].value_counts().head().to_string() if 'court' in raw_search.columns else '(no court column)')

In [ ]:
# ----------------------------------------------------------------------
# Phase B — Fetch full opinion text per cluster
#
# An opinion cluster can contain multiple opinions (majority, concurrence,
# dissent). For doctrinal extraction we want the full record, so we
# concatenate all opinions in a cluster with a separator.
#
# CHECKPOINT: per-cluster JSON cache at CACHE_OPINIONS/<cluster_id>.json
# AND a consolidated parquet at PATH_OPINION_TEXT. Per-cluster cache
# means a partial run can resume; the parquet is rebuilt from cache.
# ----------------------------------------------------------------------

def fetch_opinion_text(cluster_id):
    """Concatenate all opinions in a cluster. Tries multiple text fields
    because different cases store text in different shapes."""
    cache_file = CACHE_OPINIONS / f'{cluster_id}.json'
    if cache_file.exists():
        return json.loads(cache_file.read_text())

    url = f'{COURTLISTENER_BASE}opinions/?cluster={cluster_id}'
    data = courtlistener_get(url)
    ops = data.get('results', [])
    texts = []
    for op in ops:
        # Different opinions use different fields; try the cleanest first.
        t = (op.get('plain_text')
             or op.get('html_with_citations')
             or op.get('html')
             or op.get('xml_harvard')
             or '')
        # Crude HTML strip for the html_with_citations / html fields
        if t and ('<' in t and '>' in t):
            t = re.sub(r'<[^>]+>', ' ', t)
            t = re.sub(r'\s+', ' ', t).strip()
        if t:
            texts.append({
                'opinion_id': op.get('id'),
                'type': op.get('type'),
                'author_str': op.get('author_str'),
                'text': t,
            })
    record = {
        'cluster_id': int(cluster_id),
        'n_opinions': len(texts),
        'text': '\n\n---OPINION SEPARATOR---\n\n'.join(t['text'] for t in texts),
        'total_chars': sum(len(t['text']) for t in texts),
    }
    cache_file.write_text(json.dumps(record))
    time.sleep(REQUEST_SLEEP)
    return record


# Phase B supports two operating modes:
#   1. First run (no checkpoint parquet) — fetch everything, write the
#      parquet at the end.
#   2. Resume run (checkpoint exists) — load what's already done, fetch
#      only the missing cluster_ids, then re-write the parquet.
#
# In both modes, each cluster's response is also cached as JSON in
# CACHE_OPINIONS/<cluster_id>.json, so even within a single run a
# partial loop preserves progress. On a daily-cap 429, the loop halts
# cleanly with instructions rather than burning attempts.

if PATH_OPINION_TEXT.exists():
    print(f'Loading existing Phase B checkpoint: {PATH_OPINION_TEXT}')
    opinion_texts = pd.read_parquet(PATH_OPINION_TEXT)
    done_ids = set(opinion_texts['cluster_id'].astype(int))
    records = opinion_texts.to_dict('records')
    print(f'  resuming: {len(done_ids):,} clusters already fetched')
else:
    done_ids = set()
    records = []

# Always check for cached per-cluster JSONs that aren\'t yet in the
# parquet. The per-cluster cache is the source of truth; the parquet is
# a convenience snapshot.
cached_files = list(CACHE_OPINIONS.glob('*.json'))
print(f'  per-cluster cache files on disk: {len(cached_files):,}')

remaining = [cid for cid in unique_cases['cluster_id'].astype(int).tolist()
             if cid not in done_ids]
print(f'  clusters still to fetch: {len(remaining):,}')

halted_on_daily_cap = False
try:
    for cid in tqdm(remaining, desc='opinions'):
        try:
            records.append(fetch_opinion_text(cid))
        except DailyCapReached:
            # Re-raise so the outer try catches it; we want to halt the
            # loop, not swallow it as a per-cluster failure
            raise
        except Exception as e:
            print(f'  WARNING: failed for cluster {cid}: {e}')
            records.append({'cluster_id': int(cid), 'n_opinions': 0,
                             'text': '', 'total_chars': 0})
except DailyCapReached as e:
    halted_on_daily_cap = True
    print()
    print('=' * 70)
    print('PHASE B HALTED — CourtListener daily cap reached')
    print('=' * 70)
    print(str(e))
    print('=' * 70)

# Always flush the partial parquet so progress survives the halt
opinion_texts = pd.DataFrame(records).drop_duplicates(subset=['cluster_id'])
opinion_texts.to_parquet(PATH_OPINION_TEXT, index=False)
print(f'\nSaved partial-or-complete Phase B checkpoint: {PATH_OPINION_TEXT}')
print(f'  total clusters now in checkpoint: {len(opinion_texts):,} of {len(unique_cases):,}')

if halted_on_daily_cap:
    print('\nNext step: wait out the cooldown (Retry-After header), then re-run')
    print('this cell. It will skip already-fetched clusters and resume from')
    print('the next un-cached cluster automatically.')
    raise SystemExit('Halted on daily cap. Re-run this cell after the cooldown.')

print(f'\nPhase B complete.')
print(f'  cases with text fetched : {(opinion_texts["total_chars"] > 0).sum():,}')
print(f'  cases with empty text   : {(opinion_texts["total_chars"] == 0).sum():,}  (skip these in downstream phases)')
print(f'  total_chars distribution: '
      f'min={int(opinion_texts["total_chars"].min())}, '
      f'median={int(opinion_texts["total_chars"].median())}, '
      f'max={int(opinion_texts["total_chars"].max())}')

In [ ]:
# ----------------------------------------------------------------------
# Phase C (part 1) — Passage extraction
#
# Pre-extract passages from each opinion that mention doctrine keywords.
# This keeps the LLM extraction step (Phase D) cheap and focused: it
# only sees the relevant snippets, not the full opinion.
#
# The pattern catches scènes à faire, stock language, merger doctrine,
# and the standard "indispensable / necessarily follows" phrases courts
# use to identify unprotectable material.
#
# CHECKPOINT: PATH_PASSAGES
# ----------------------------------------------------------------------

DOCTRINE_PATTERN = re.compile(
    r'(?P<head>sc[èe]nes?\s+[àa]\s+faire'
    r'|stock\s+(?:element|character|feature|scene|figure)'
    r'|unprotect\w+'
    r'|standard\s+(?:feature|element|character)'
    r'|merger\s+doctrine'
    r'|indispensable'
    r'|necessarily\s+follow\w*'
    r'|dictated\s+by'
    r'|inherent\s+(?:in|to)\s+the\s+(?:genre|story|theme))'
    r'(?P<body>.{0,1200})',
    re.IGNORECASE | re.DOTALL,
)


def extract_passages_from_text(text, max_passages=8, max_chars_each=1200):
    """Return a list of unique passage strings, sliced around each
    doctrine-keyword hit. De-duplicates near-identical passages so the
    LLM doesn't see the same snippet three times."""
    if not text:
        return []
    matches = list(DOCTRINE_PATTERN.finditer(text))
    passages = []
    seen_starts = set()
    for m in matches:
        start = m.start()
        # avoid near-duplicate passages from clustered keyword hits
        if any(abs(start - s) < 400 for s in seen_starts):
            continue
        seen_starts.add(start)
        passages.append(m.group(0)[:max_chars_each])
        if len(passages) >= max_passages:
            break
    return passages


if PATH_PASSAGES.exists():
    print(f'Loading existing Phase C checkpoint (passages): {PATH_PASSAGES}')
    passages_df = pd.read_parquet(PATH_PASSAGES)
else:
    print(f'Running Phase C part 1: extracting doctrine passages...')
    records = []
    for _, row in tqdm(opinion_texts.iterrows(),
                       total=len(opinion_texts), desc='passages'):
        passages = extract_passages_from_text(row['text'])
        records.append({
            'cluster_id': int(row['cluster_id']),
            'passages': passages,
            'passage_count': len(passages),
        })
    passages_df = pd.DataFrame(records)
    passages_df.to_parquet(PATH_PASSAGES, index=False)
    print(f'Saved: {PATH_PASSAGES}')

print(f'\nPhase C part 1 complete.')
print(f'  cases with ≥1 doctrine passage : {(passages_df["passage_count"] > 0).sum():,}')
print(f'  cases with 0 doctrine passages : {(passages_df["passage_count"] == 0).sum():,}')
print(f'  median passages per case       : {int(passages_df["passage_count"].median())}')
print(f'  max passages per case          : {int(passages_df["passage_count"].max())}')

In [ ]:
# ----------------------------------------------------------------------
# Phase C (part 2) — Auto-triage cases by medium and scope
#
# For each case with ≥1 doctrine passage, ask the LLM to classify the
# medium of the disputed work and assign a scope tag and a relevant
# canonical genre. The output of this cell is an UNCURATED triage table
# that you (Mira) will manually review in the next cell.
#
# CHECKPOINT: PATH_TRIAGE_AUTO
# ----------------------------------------------------------------------

TRIAGE_SYSTEM = """You are reading a published US copyright opinion that applies scènes à faire or merger doctrine. Classify the medium of the disputed creative work and assign a scope tag.

MEDIUM (the kind of creative work the dispute is about):
- film: a motion picture or feature film
- TV: television series or episode
- music: songs, album, composition, lyrics, or musical theater
- literature: novel, short story, play, or other narrative writing
- architecture: architectural design or floor plans
- art: visual art, decorative art, sculpture, illustration
- photography: photographic works
- fashion: clothing, jewelry, or fashion design
- software: software UI, code, or other functional works
- other: anything not covered above

For music cases ONLY, additionally classify the MUSIC SUBTYPE:
- narrative_musical: the dispute concerns the plot, characters, or story of a musical film, musical, biopic, or band-narrative film
- pure_composition: the dispute concerns chord progressions, rhythmic patterns, melodies, or lyric phrasings as such
- null: not a music case

SCOPE TAG:
- in_scope: the work is film, TV drama, literature, or a narrative_musical music case — disputes the framework's movie-plot corpus can address
- adjacent: the work is pure_composition music — the framework's Music genre is built from films about music, not from compositions, so the validation can only partially apply
- out_of_scope: the work is architecture, art, photography, fashion, software, or other non-narrative material — the framework's movie-plot corpus does not apply

RELEVANT GENRE (one of the seventeen canonical genres, or 'unknown'):
Action, Adventure, Animation, Comedy, Crime, Drama, Family, Fantasy, History, Horror, Music, Mystery, Romance, Science Fiction, Thriller, War, Western.
Pick the SINGLE most-applicable genre for the disputed work. If the work spans multiple genres, pick the most specific one applicable to what the court was assessing. If you cannot tell, return 'unknown'.

Return JSON ONLY:
{
  "medium": "<one of the above>",
  "music_subtype": "<one of the above or null>",
  "scope_tag": "<in_scope|adjacent|out_of_scope>",
  "relevant_genre": "<canonical genre or 'unknown'>",
  "scope_rationale": "<one short sentence>"
}
"""


def auto_triage_case(case_row, passages):
    if not passages:
        return {
            'medium': 'unknown', 'music_subtype': None,
            'scope_tag': 'out_of_scope', 'relevant_genre': 'unknown',
            'scope_rationale': 'no doctrine passages found in opinion text',
        }
    user_prompt = (
        f"CASE NAME: {case_row.get('caseName', '')}\n"
        f"COURT: {case_row.get('court', '')}\n"
        f"DATE: {case_row.get('dateFiled', '')}\n\n"
        f"PASSAGES (separated by ---):\n"
        + '\n---\n'.join(passages[:3])
    )
    resp = anthropic_client.messages.create(
        model=MODEL_NAME,
        max_tokens=400,
        temperature=0,
        system=[{'type':'text','text':TRIAGE_SYSTEM,'cache_control':{'type':'ephemeral'}}],
        messages=[{'role':'user','content':user_prompt}],
    )
    text = ''.join(b.text for b in resp.content if getattr(b,'type',None)=='text')
    try:
        return json.loads(text[text.find('{'):text.rfind('}')+1])
    except json.JSONDecodeError:
        return {
            'medium': 'unknown', 'music_subtype': None,
            'scope_tag': 'out_of_scope', 'relevant_genre': 'unknown',
            'scope_rationale': f'JSON parse failed; raw response: {text[:200]}',
        }


if PATH_TRIAGE_AUTO.exists():
    print(f'Loading existing Phase C checkpoint (triage): {PATH_TRIAGE_AUTO}')
    triage_auto = pd.read_parquet(PATH_TRIAGE_AUTO)
else:
    print('Running Phase C part 2: auto-triaging cases...')
    case_meta = unique_cases.set_index('cluster_id')
    passage_lookup = passages_df.set_index('cluster_id')

    records = []
    for cid, prow in tqdm(passage_lookup.iterrows(), total=len(passage_lookup), desc='triage'):
        cid_int = int(cid)
        case_row = case_meta.loc[cid_int] if cid_int in case_meta.index else {}
        passages = list(prow['passages']) if hasattr(prow['passages'], '__iter__') else []
        triage = auto_triage_case(case_row, passages)
        records.append({
            'cluster_id': cid_int,
            'caseName': case_row.get('caseName', '') if hasattr(case_row, 'get') else '',
            'court': case_row.get('court', '') if hasattr(case_row, 'get') else '',
            'dateFiled': case_row.get('dateFiled', '') if hasattr(case_row, 'get') else '',
            'passage_count': prow['passage_count'],
            **triage,
            'manually_verified': False,
        })
    triage_auto = pd.DataFrame(records)
    triage_auto.to_parquet(PATH_TRIAGE_AUTO, index=False)
    print(f'Saved: {PATH_TRIAGE_AUTO}')

print(f'\nPhase C part 2 complete.')
print(f'  scope distribution:')
print(triage_auto['scope_tag'].value_counts().to_string())
print(f'\n  medium distribution:')
print(triage_auto['medium'].value_counts().to_string())
print(f'\n  music subtypes (where applicable):')
print(triage_auto[triage_auto['medium']=='music']['music_subtype'].value_counts(dropna=False).to_string())
print(f'\n  in-scope cases with a canonical genre assigned:')
in_scope = triage_auto[triage_auto['scope_tag']=='in_scope']
print(in_scope['relevant_genre'].value_counts().to_string())

## Manual review checkpoint — case triage

**STOP HERE.** Before the notebook proceeds, you (Mira) need to review the auto-triage assignments.

### What to do

1. The auto-triage just wrote `case_triage_auto.parquet` to the output directory. Convert it to a CSV you can read in Google Sheets or Excel:
   ```python
   triage_auto.to_csv(OUT_DIR / 'case_triage_REVIEW.csv', index=False)
   ```
   (Or just run the cell below, which does this for you.)

2. Open `case_triage_REVIEW.csv` in your editor of choice.

3. For each case, verify the auto-assigned `medium`, `music_subtype`, `scope_tag`, and `relevant_genre`. **Spot-check at least 20% of all cases manually**, and **every music case** (the in-scope/adjacent distinction matters more for music than anywhere else).

4. **The judgment calls you need to make:**

   - **Medium**: is the disputed work really a film/TV/literature work, or is it something else the LLM mislabeled? Read the case name and the court if you're unsure.
   - **Music subtype**: a musical film's plot dispute is `narrative_musical` (in_scope). A song-composition or chord-progression dispute is `pure_composition` (adjacent). When in doubt, look at the case name and search for it online.
   - **Scope tag**: confirm the routing follows the per-case rationale. If the LLM tagged a case `out_of_scope` but you think the framework could partially address it, change to `adjacent` and write a one-line `scope_rationale`.
   - **Relevant genre**: most cases involve a clear single genre. For genuinely cross-genre cases, pick the most specific applicable one. If you cannot tell from the case name and snippet, leave `unknown` and tag `manually_verified=True` with a note.

5. For every row you reviewed, set `manually_verified = True`. Cases you did NOT manually check stay at `False` (the auto-triage stands).

6. Save your reviewed file as `case_triage_CURATED.csv` (same directory, change the suffix). Then run the next cell to load it back into the notebook.

### Questions to flag for David

- Cases where you genuinely cannot determine the medium or scope. Note them in a `notes` column.
- Cases where the auto-triage assigned a genre that does not match what you read in the opinion (e.g., the case is about a romantic comedy but the LLM tagged it `Crime`). Note the corrections.
- Any cases that span multiple genres in a doctrinally interesting way (a horror-romance, a science-fiction-western). David may want to handle these as cross-genre validation cases.

In [ ]:
# ----------------------------------------------------------------------
# Export the auto-triage to CSV for your review, then load your curated
# version back into the notebook.
# ----------------------------------------------------------------------

# Step 1: write the auto-triage to CSV (if not already done)
review_csv = OUT_DIR / 'case_triage_REVIEW.csv'
if not review_csv.exists():
    triage_auto.to_csv(review_csv, index=False)
    print(f'Wrote review CSV: {review_csv}')
    print(f'Open this file in your editor, review the rows, set manually_verified=True')
    print(f'on the rows you checked, and save as case_triage_CURATED.csv in the same directory.')
else:
    print(f'Review CSV already exists at {review_csv}.')

# Step 2: try to load the curated file. If it does not exist yet, we
# proceed with the auto-triage and note that no manual review has happened.
curated_csv = OUT_DIR / 'case_triage_CURATED.csv'
if curated_csv.exists():
    triage = pd.read_csv(curated_csv)
    # Boolean coercion for manually_verified — CSV can stringify it
    triage['manually_verified'] = triage['manually_verified'].astype(str).str.lower().isin(['true','1','yes'])
    triage.to_parquet(PATH_TRIAGE_CURATED, index=False)
    print(f'\nLoaded curated triage: {len(triage):,} rows, '
          f'{int(triage["manually_verified"].sum())} manually verified')
else:
    print(f'\n** NOTE ** No curated CSV found at {curated_csv}.')
    print(f'Proceeding with auto-triage only. Re-run this cell after curating.')
    triage = triage_auto.copy()
    triage.to_parquet(PATH_TRIAGE_CURATED, index=False)

print(f'\nFinal triage breakdown:')
print(f'  in_scope     : {(triage["scope_tag"]=="in_scope").sum()}')
print(f'  adjacent     : {(triage["scope_tag"]=="adjacent").sum()}')
print(f'  out_of_scope : {(triage["scope_tag"]=="out_of_scope").sum()}')

In [ ]:
# ----------------------------------------------------------------------
# Phase D — LLM extraction of judicially-identified stock elements
#
# For each in_scope and adjacent case, send the doctrine passages to
# Claude with a structured-output prompt asking for the specific stock
# elements (and combinations) the judge identified.
#
# CHECKPOINT: per-cluster JSON in CACHE_EXTRACTION/ AND consolidated
# parquet at PATH_EXTRACT_AUTO.
# ----------------------------------------------------------------------

EXTRACT_SYSTEM = """You are reading excerpts from copyright opinions where a court applied the scènes à faire or merger doctrine. Your task is to extract the specific narrative elements the court identified as stock, standard, customary, stereotypical, common, indispensable, dictated by genre/history/idea, or otherwise unprotectable.

Distinguish:
- SINGLE ELEMENTS: a character type (sheriff, mad scientist), a setting (frontier town, haunted house), an object (six-shooter, magic wand), an event (shootout, wedding), a theme (revenge, redemption), or a relationship (forbidden love, mentor-student).
- COMBINATIONS: a specific co-occurring pair or group of elements the court treated as a unit (e.g. "Nazis seeking a buried magical artifact", "a hospital drama with dying patients and harried doctors").

Return JSON ONLY in this exact schema:
{
  "case": "<case name as given>",
  "relevant_genre": "<one of the 17 canonical genres or 'unknown'>",
  "single_elements": [
    {"element": "<lowercase noun/verb/short phrase>", "element_type_hint": "<character_type|setting|object|event|theme|relationship>", "passage_quote": "<short verbatim quote>"}
  ],
  "combinations": [
    {"elements": ["<element1>", "<element2>", ...], "passage_quote": "<short verbatim quote>"}
  ],
  "notes": "<any caveat or ambiguity, or empty string>"
}

If the opinion identifies no specific elements (only general doctrinal language about scènes à faire without naming items), return empty lists for single_elements and combinations.

Be CONSERVATIVE. Only include items the judge explicitly identified, not items you might infer."""


def extract_elements_for_case(cluster_id, case_name, passages):
    cache_file = CACHE_EXTRACTION / f'{int(cluster_id)}.json'
    if cache_file.exists():
        return json.loads(cache_file.read_text())

    if not passages:
        result = {'case': case_name, 'relevant_genre': 'unknown',
                  'single_elements': [], 'combinations': [], 'notes': 'no passages'}
        cache_file.write_text(json.dumps(result))
        return result

    user_prompt = f"""CASE: {case_name}

PASSAGES (separated by ---):
{chr(10).join('---' + chr(10) + p for p in passages[:6])}"""

    resp = anthropic_client.messages.create(
        model=MODEL_NAME,
        max_tokens=EXTRACTION_MAX_TOKENS,
        temperature=0,
        system=[{'type':'text','text':EXTRACT_SYSTEM,'cache_control':{'type':'ephemeral'}}],
        messages=[{'role':'user','content':user_prompt}],
    )
    text = ''.join(b.text for b in resp.content if getattr(b,'type',None)=='text')
    try:
        result = json.loads(text[text.find('{'):text.rfind('}')+1])
    except json.JSONDecodeError:
        result = {'case': case_name, 'relevant_genre': 'unknown',
                  'single_elements': [], 'combinations': [],
                  'notes': f'JSON parse failed; raw: {text[:300]}'}
    cache_file.write_text(json.dumps(result))
    return result


if PATH_EXTRACT_AUTO.exists():
    print(f'Loading existing Phase D checkpoint: {PATH_EXTRACT_AUTO}')
    extract_auto = pd.read_parquet(PATH_EXTRACT_AUTO)
else:
    print('Running Phase D: LLM extraction of judicial elements...')
    in_scope_or_adjacent = triage[triage['scope_tag'].isin(['in_scope','adjacent'])]
    passage_lookup = passages_df.set_index('cluster_id')

    rows = []
    for _, t in tqdm(in_scope_or_adjacent.iterrows(),
                     total=len(in_scope_or_adjacent), desc='extract'):
        cid = int(t['cluster_id'])
        passages = list(passage_lookup.loc[cid]['passages']) if cid in passage_lookup.index else []
        result = extract_elements_for_case(cid, t['caseName'], passages)

        for se in result.get('single_elements', []):
            rows.append({
                'cluster_id': cid,
                'caseName': t['caseName'],
                'scope_tag': t['scope_tag'],
                'relevant_genre': t['relevant_genre'],
                'kind': 'single',
                'element': se.get('element', '').strip().lower(),
                'element_type_hint': se.get('element_type_hint', ''),
                'combination_id': None,
                'combination_size': None,
                'combination_member_index': None,
                'passage_quote': se.get('passage_quote', ''),
                'notes': result.get('notes', ''),
                'manually_verified': False,
            })
        for ci, combo in enumerate(result.get('combinations', [])):
            elements = combo.get('elements', [])
            for mi, elem in enumerate(elements):
                rows.append({
                    'cluster_id': cid,
                    'caseName': t['caseName'],
                    'scope_tag': t['scope_tag'],
                    'relevant_genre': t['relevant_genre'],
                    'kind': 'combination_member',
                    'element': elem.strip().lower(),
                    'element_type_hint': '',
                    'combination_id': ci,
                    'combination_size': len(elements),
                    'combination_member_index': mi,
                    'passage_quote': combo.get('passage_quote', ''),
                    'notes': result.get('notes', ''),
                    'manually_verified': False,
                })
        # cases with zero extractions still get a row so we can see them
        if not result.get('single_elements') and not result.get('combinations'):
            rows.append({
                'cluster_id': cid, 'caseName': t['caseName'],
                'scope_tag': t['scope_tag'], 'relevant_genre': t['relevant_genre'],
                'kind': 'no_extraction', 'element': '', 'element_type_hint': '',
                'combination_id': None, 'combination_size': None,
                'combination_member_index': None,
                'passage_quote': '', 'notes': result.get('notes', ''),
                'manually_verified': False,
            })
    extract_auto = pd.DataFrame(rows)
    extract_auto.to_parquet(PATH_EXTRACT_AUTO, index=False)
    print(f'Saved: {PATH_EXTRACT_AUTO}')

print(f'\nPhase D complete.')
n_cases_with_singles = extract_auto[extract_auto['kind']=='single']['cluster_id'].nunique()
n_cases_with_combos  = extract_auto[extract_auto['kind']=='combination_member']['cluster_id'].nunique()
n_cases_no_extract   = extract_auto[extract_auto['kind']=='no_extraction']['cluster_id'].nunique()
print(f'  cases with single elements   : {n_cases_with_singles}')
print(f'  cases with combinations      : {n_cases_with_combos}')
print(f'  cases with no extraction     : {n_cases_no_extract}')
print(f'  total single elements        : {(extract_auto["kind"]=="single").sum()}')
print(f'  total combinations           : {extract_auto[extract_auto["kind"]=="combination_member"].groupby(["cluster_id","combination_id"]).ngroups}')

## Manual review checkpoint — judicial element extraction

**STOP HERE.** Before the notebook proceeds, you (Mira) need to review the LLM extractions.

### What to do

1. The previous cell wrote `judicial_elements_auto.parquet` to the output directory. The next cell will export it to a CSV for your review.

2. Open `judicial_elements_REVIEW.csv` in Google Sheets or Excel.

3. For each row, verify that the LLM-extracted element is one the judge actually identified as stock or unprotectable. **Read the `passage_quote` column** — the LLM should be quoting the judge's identification verbatim. If the quote does not actually identify the element as stock (the LLM hallucinated, or it captured an element the judge mentioned but did not flag), delete the row or mark it for exclusion.

4. **Edit rules:**

   - To **accept** a row as ground truth: set `manually_verified = True`.
   - To **delete** a row: delete the row in the CSV. (The next cell loads only rows present in the CURATED CSV.)
   - To **correct** a row (e.g., the LLM extracted "sheriff" but the judge said "frontier sheriff"): edit the `element` column.
   - To **add** an element the LLM missed: append a new row with `manually_verified=True`. Use `kind=single` for individual elements, `kind=combination_member` for combinations (give all members of a combination the same `combination_id` and increment `combination_member_index`).

5. **Watch for these LLM failure modes:**

   - Capturing elements the judge mentioned but did NOT flag as stock. Quote check.
   - Combining single elements into pseudo-combinations the judge didn't make as a combination claim. Quote check.
   - Missing combinations that span multiple sentences in the opinion. You may need to add these.
   - Hallucinated `relevant_genre` (the LLM may guess "Drama" by default). If the judge identifies stock elements without naming a genre, leave the genre as `unknown` and David will decide whether to assign one downstream.

6. Save your reviewed file as `judicial_elements_CURATED.csv` and run the next cell.

### Estimated time

A few hours for ~100 cases. The first 20 will feel slow. By the time you reach 50, you'll have a rhythm. The output of this step is the ground truth the entire validation rests on, so take the time.

### When to ask David

- Cases where the judge's identification is genuinely ambiguous (the opinion uses doctrine language but does not name specific elements).
- Cases where the genre attribution from triage looks wrong in light of the actual element list.
- Cases where the judicial combination is more than three elements and decomposing into pairs feels like it loses meaning.

In [ ]:
# ----------------------------------------------------------------------
# Export the LLM extractions to CSV, then load your curated version.
# Same pattern as the triage review.
# ----------------------------------------------------------------------

extract_csv = OUT_DIR / 'judicial_elements_REVIEW.csv'
if not extract_csv.exists():
    extract_auto.to_csv(extract_csv, index=False)
    print(f'Wrote review CSV: {extract_csv}')
    print(f'Open in your editor, review row-by-row, save as judicial_elements_CURATED.csv')
else:
    print(f'Review CSV already exists at {extract_csv}.')

curated_extract_csv = OUT_DIR / 'judicial_elements_CURATED.csv'
if curated_extract_csv.exists():
    judicial = pd.read_csv(curated_extract_csv)
    judicial['manually_verified'] = judicial['manually_verified'].astype(str).str.lower().isin(['true','1','yes'])
    judicial.to_parquet(PATH_EXTRACT_CURATED, index=False)
    print(f'\nLoaded curated extraction: {len(judicial):,} rows, '
          f'{int(judicial["manually_verified"].sum())} manually verified')
else:
    print(f'\n** NOTE ** No curated CSV found at {curated_extract_csv}.')
    print(f'Proceeding with auto-extraction only. Re-run this cell after curating.')
    judicial = extract_auto.copy()
    judicial.to_parquet(PATH_EXTRACT_CURATED, index=False)

# Filter to in_scope and adjacent cases with actual element extractions
judicial_active = judicial[
    (judicial['scope_tag'].isin(['in_scope','adjacent'])) &
    (judicial['kind'].isin(['single','combination_member'])) &
    (judicial['element'].fillna('').str.strip() != '')
].copy()
judicial_active['element_norm'] = judicial_active['element'].str.strip().str.lower()
print(f'\nActive judicial identifications (single + combination members): {len(judicial_active):,}')
print(f'  by kind:')
print(judicial_active['kind'].value_counts().to_string())
print(f'  by relevant_genre:')
print(judicial_active['relevant_genre'].value_counts().to_string())

In [ ]:
# ----------------------------------------------------------------------
# Phase E (part 1) — Match judicial single elements to the pipeline
#
# For each judicially-identified single element in a relevant genre,
# search stockiness_long_classified for the element and record its
# rank, percentile, ensemble score, and classification.
#
# Matching cascade:
#   1. Exact lowercase string match
#   2. Substring match (e.g. "frontier town" → matches the chunk "frontier town")
#   3. Fuzzy match via rapidfuzz token_sort_ratio ≥ 85, restricted to the
#      element_type_hint if provided
#
# CHECKPOINT: PATH_ELEM_MATCHES
# ----------------------------------------------------------------------

# Pre-index stockiness_long_classified by genre for fast filtered lookup
pipeline_by_genre = {g: df.copy() for g, df in stk_cls.groupby('genre')}


def candidate_element_types(hint):
    """Map the LLM's element_type_hint to the spaCy-pipeline element_types
    we should look in. Hint is one of character_type|setting|object|event|
    theme|relationship; we map each to noun and chunk (and verb for event)
    because the spaCy pipeline doesn't distinguish the LLM's categories at
    extraction time. The 03b taxonomy column on the matched row will tell
    us whether it agrees with the hint."""
    base = ['noun', 'chunk']
    if hint == 'event':
        base.append('verb')
    if hint in ('character_type',):
        base.append('entity_NORP')
    return base


def match_single_element(judicial_elem, genre, hint=None):
    """Return the best-matching row from stockiness_long_classified, or
    None if no good match is found. Tries exact, substring, fuzzy in order."""
    if genre not in pipeline_by_genre:
        return None
    sub = pipeline_by_genre[genre]
    e_norm = str(judicial_elem).strip().lower()
    if not e_norm:
        return None
    types_to_try = candidate_element_types(hint) if hint else None

    # 1. Exact match
    cand = sub[sub['element'] == e_norm]
    if types_to_try is not None:
        cand_typed = cand[cand['element_type'].isin(types_to_try)]
        if not cand_typed.empty:
            cand = cand_typed
    if not cand.empty:
        row = cand.sort_values('ensemble', ascending=False).iloc[0]
        return {'method': 'exact', 'row': row}

    # 2. Substring (one direction: judicial element appears IN the pipeline element)
    cand = sub[sub['element'].str.contains(re.escape(e_norm), regex=True, na=False)]
    if types_to_try is not None:
        cand_typed = cand[cand['element_type'].isin(types_to_try)]
        if not cand_typed.empty:
            cand = cand_typed
    if not cand.empty:
        row = cand.sort_values('ensemble', ascending=False).iloc[0]
        return {'method': 'substring', 'row': row}

    # 3. Fuzzy (rapidfuzz) — restrict to the type hint if given to keep
    # the candidate pool reasonable
    pool = sub
    if types_to_try is not None:
        pool = sub[sub['element_type'].isin(types_to_try)]
    candidates = pool['element'].tolist()
    if not candidates:
        return None
    match = process.extractOne(e_norm, candidates, scorer=fuzz.token_sort_ratio, score_cutoff=85)
    if match is None:
        return None
    matched_string, score, idx = match
    row = pool.iloc[idx]
    return {'method': f'fuzzy_{int(score)}', 'row': row}


if PATH_ELEM_MATCHES.exists():
    print(f'Loading existing Phase E checkpoint (elements): {PATH_ELEM_MATCHES}')
    elem_matches = pd.read_parquet(PATH_ELEM_MATCHES)
else:
    print('Running Phase E part 1: matching judicial single elements to pipeline...')
    rows = []
    single_judicials = judicial_active[judicial_active['kind']=='single']
    for _, j in tqdm(single_judicials.iterrows(),
                     total=len(single_judicials), desc='element match'):
        out = {
            'cluster_id': j['cluster_id'],
            'caseName': j['caseName'],
            'judicial_element': j['element_norm'],
            'element_type_hint': j['element_type_hint'],
            'relevant_genre': j['relevant_genre'],
            'passage_quote': j['passage_quote'],
        }
        if j['relevant_genre'] not in pipeline_by_genre:
            out.update({'in_pipeline': False, 'match_method': None})
        else:
            match = match_single_element(j['element_norm'], j['relevant_genre'], j['element_type_hint'])
            if match is None:
                out.update({'in_pipeline': False, 'match_method': None})
            else:
                row = match['row']
                rank = int(row['rank_in_genre_by_type'])
                pct = float(row['percentile_in_genre_by_type'])
                out.update({
                    'in_pipeline': True,
                    'match_method': match['method'],
                    'match_element': row['element'],
                    'match_element_type': row['element_type'],
                    'element_taxonomy': row['element_taxonomy'],
                    'is_narrative_element': bool(row['is_narrative_element']),
                    'ensemble_score': float(row['ensemble']),
                    'ensemble_rank_in_genre_by_type': rank,
                    'ensemble_percentile_in_genre_by_type': pct,
                    'flagged_high_stockiness': pct >= HIGH_STOCKINESS_PERCENTILE,
                })
        rows.append(out)
    elem_matches = pd.DataFrame(rows)
    elem_matches.to_parquet(PATH_ELEM_MATCHES, index=False)
    print(f'Saved: {PATH_ELEM_MATCHES}')

print(f'\nPhase E part 1 complete.')
print(f'  judicial single elements        : {len(elem_matches)}')
print(f'  matched in pipeline             : {int(elem_matches["in_pipeline"].sum())}')
print(f'  flagged high stockiness         : {int(elem_matches.get("flagged_high_stockiness", pd.Series([])).sum())}')
print(f'  match method distribution       :')
print(elem_matches['match_method'].value_counts(dropna=False).to_string())

In [ ]:
# ----------------------------------------------------------------------
# Phase E (part 2) — Match judicial combinations to pipeline pairs
#
# For each multi-element judicial combination, decompose into all
# unordered pairs and look each pair up in combination_stockiness for
# the relevant genre. Report per-pair stats AND a combination-level
# summary (min joint_count, min combination_stock, any_flagged_top5).
#
# CHECKPOINT: PATH_COMB_MATCHES
# ----------------------------------------------------------------------

# Pre-index combination_stockiness by genre for fast filtered lookup
comb_by_genre = {g: df.copy() for g, df in comb_stock.groupby('genre')}


def find_pair_in_combinations(elem_i, elem_j, genre):
    """Look up a pair (regardless of order) in combination_stockiness for
    the given genre. Returns the row or None."""
    if genre not in comb_by_genre:
        return None
    sub = comb_by_genre[genre]
    # try both orderings (i, j) and (j, i)
    cand = sub[
        ((sub['element_i'] == elem_i) & (sub['element_j'] == elem_j)) |
        ((sub['element_i'] == elem_j) & (sub['element_j'] == elem_i))
    ]
    if cand.empty:
        return None
    return cand.iloc[0]


if PATH_COMB_MATCHES.exists():
    print(f'Loading existing Phase E checkpoint (combinations): {PATH_COMB_MATCHES}')
    comb_matches = pd.read_parquet(PATH_COMB_MATCHES)
else:
    print('Running Phase E part 2: matching judicial combinations to pipeline pairs...')
    pair_rows = []
    combo_summary_rows = []

    combo_members = judicial_active[judicial_active['kind']=='combination_member']
    grouped = combo_members.groupby(['cluster_id','combination_id'])

    for (cid, combo_id), grp in tqdm(grouped, desc='combinations'):
        elements = [e for e in grp['element_norm'].tolist() if e]
        if len(elements) < 2:
            continue
        genre = grp['relevant_genre'].iloc[0]
        case_name = grp['caseName'].iloc[0]
        # Decompose into all unordered pairs
        per_pair_results = []
        for i in range(len(elements)):
            for j in range(i+1, len(elements)):
                row = find_pair_in_combinations(elements[i], elements[j], genre)
                if row is None:
                    per_pair_results.append({
                        'pair_in_pipeline': False,
                        'elem_i': elements[i], 'elem_j': elements[j],
                    })
                    pair_rows.append({
                        'cluster_id': cid, 'caseName': case_name,
                        'combination_id': combo_id,
                        'judicial_elem_i': elements[i],
                        'judicial_elem_j': elements[j],
                        'relevant_genre': genre,
                        'pair_in_pipeline': False,
                    })
                else:
                    per_pair_results.append({
                        'pair_in_pipeline': True,
                        'joint_count_g': int(row['joint_count_g']),
                        'lift': float(row['lift']),
                        'combination_stock': float(row['combination_stock']),
                        'combination_stock_percentile': float(row['combination_stock_percentile']),
                        'posterior_smoothed': float(row['posterior_smoothed']),
                        'z_ensemble_i': float(row['z_ensemble_i']),
                        'z_ensemble_j': float(row['z_ensemble_j']),
                    })
                    pair_rows.append({
                        'cluster_id': cid, 'caseName': case_name,
                        'combination_id': combo_id,
                        'judicial_elem_i': elements[i],
                        'judicial_elem_j': elements[j],
                        'relevant_genre': genre,
                        'pair_in_pipeline': True,
                        'joint_count_g': int(row['joint_count_g']),
                        'lift': float(row['lift']),
                        'combination_stock': float(row['combination_stock']),
                        'combination_stock_percentile': float(row['combination_stock_percentile']),
                        'posterior_smoothed': float(row['posterior_smoothed']),
                        'z_ensemble_i': float(row['z_ensemble_i']),
                        'z_ensemble_j': float(row['z_ensemble_j']),
                        'flagged_top1':  float(row['combination_stock_percentile']) >= COMBINATION_TOP_STRICT,
                        'flagged_top5':  float(row['combination_stock_percentile']) >= COMBINATION_TOP_HEADLINE,
                        'flagged_top10': float(row['combination_stock_percentile']) >= COMBINATION_TOP_GENEROUS,
                    })

        # Combination-level summary
        matched_pairs = [r for r in per_pair_results if r['pair_in_pipeline']]
        combo_summary_rows.append({
            'cluster_id': cid, 'caseName': case_name,
            'combination_id': combo_id,
            'elements': '|'.join(elements),
            'n_elements': len(elements),
            'n_pairs': len(per_pair_results),
            'n_pairs_in_pipeline': len(matched_pairs),
            'min_joint_count_g': min((r['joint_count_g'] for r in matched_pairs), default=None),
            'mean_combination_stock': (sum(r['combination_stock'] for r in matched_pairs) / len(matched_pairs)) if matched_pairs else None,
            'min_combination_stock': min((r['combination_stock'] for r in matched_pairs), default=None),
            'any_flagged_top1':  any(r.get('combination_stock_percentile', 0) >= COMBINATION_TOP_STRICT for r in matched_pairs),
            'any_flagged_top5':  any(r.get('combination_stock_percentile', 0) >= COMBINATION_TOP_HEADLINE for r in matched_pairs),
            'any_flagged_top10': any(r.get('combination_stock_percentile', 0) >= COMBINATION_TOP_GENEROUS for r in matched_pairs),
            'relevant_genre': genre,
        })

    comb_matches = pd.DataFrame(pair_rows)
    comb_summary = pd.DataFrame(combo_summary_rows)
    comb_matches.to_parquet(PATH_COMB_MATCHES, index=False)
    comb_summary.to_parquet(OUT_DIR / 'combination_summary.parquet', index=False)
    print(f'Saved: {PATH_COMB_MATCHES}')

print(f'\nPhase E part 2 complete.')
if len(comb_matches):
    print(f'  judicial pair lookups            : {len(comb_matches)}')
    print(f'  pairs in pipeline                : {int(comb_matches["pair_in_pipeline"].sum())}')
    if 'flagged_top5' in comb_matches.columns:
        print(f'  pairs flagged top 5%             : {int(comb_matches["flagged_top5"].sum())}')
        print(f'  pairs flagged top 1% (strict)    : {int(comb_matches["flagged_top1"].sum())}')
        print(f'  pairs flagged top 10% (generous) : {int(comb_matches["flagged_top10"].sum())}')
else:
    print('  No judicial combinations to match (or all combinations had <2 elements).')

In [ ]:
# ----------------------------------------------------------------------
# Phase F — Validation metrics and report
#
# Compute P@K, R@K, F1@K at K=10, 25, 50, 100 per genre and corpus-wide.
# Compute Spearman ρ per genre. Build the validation table and write
# the report.
# ----------------------------------------------------------------------

from scipy.stats import spearmanr


def precision_recall_at_k(judicial_elements_in_genre, pipeline_top_k_in_genre):
    """Both args are sets of element strings (lowercased). Returns p, r, f1."""
    if not pipeline_top_k_in_genre:
        return 0.0, 0.0, 0.0
    hits = judicial_elements_in_genre & pipeline_top_k_in_genre
    p = len(hits) / len(pipeline_top_k_in_genre)
    r = len(hits) / len(judicial_elements_in_genre) if judicial_elements_in_genre else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    return p, r, f1


# Build per-genre judicial element sets (in_scope only for headline numbers)
elem_matches_in_scope = elem_matches[
    elem_matches['caseName'].isin(triage[triage['scope_tag']=='in_scope']['caseName'])
] if 'caseName' in elem_matches.columns else elem_matches

judicial_by_genre = defaultdict(set)
for _, row in elem_matches.iterrows():
    if row['relevant_genre'] in CANONICAL_GENRES:
        judicial_by_genre[row['relevant_genre']].add(row['judicial_element'])

# Build the per-genre pipeline top-K sets (over narrative elements only,
# pooled across element_types because judicial identifications don't
# correspond cleanly to a single element_type)
def pipeline_top_k(genre, k):
    if genre not in pipeline_by_genre:
        return set()
    sub = pipeline_by_genre[genre]
    narrative = sub[sub['is_narrative_element']]
    top = narrative.nlargest(k, 'ensemble')
    return set(top['element'].str.lower().tolist())

# Per-genre precision/recall/F1 at each K
per_genre_pr = []
for genre in CANONICAL_GENRES:
    if not judicial_by_genre[genre]:
        continue
    for K in K_VALUES:
        p, r, f1 = precision_recall_at_k(
            judicial_by_genre[genre],
            pipeline_top_k(genre, K),
        )
        per_genre_pr.append({'genre': genre, 'K': K, 'precision': p,
                             'recall': r, 'f1': f1,
                             'n_judicial_in_genre': len(judicial_by_genre[genre])})
per_genre_pr_df = pd.DataFrame(per_genre_pr)

# Corpus-wide aggregate (unweighted mean across genres with ≥1 judicial id)
corpus_pr = (per_genre_pr_df.groupby('K')[['precision','recall','f1']]
             .mean().reset_index())
print('Corpus-wide P@K / R@K / F1@K (unweighted mean over genres with ≥1 judicial identification):')
print(corpus_pr.to_string(index=False))
print()
print('Per-genre P@K / R@K / F1@K (full curve):')
print(per_genre_pr_df.to_string(index=False))

# Spearman rank correlation per genre. For each genre, build a vector of
# (binary judicial label, pipeline ensemble) for all narrative elements,
# then compute ρ.
spearman_rows = []
for genre in CANONICAL_GENRES:
    if not judicial_by_genre[genre]:
        continue
    sub = pipeline_by_genre.get(genre, pd.DataFrame()).copy()
    narrative = sub[sub['is_narrative_element']]
    narrative = narrative.drop_duplicates('element')
    judicial_set = judicial_by_genre[genre]
    narrative['is_judicial'] = narrative['element'].str.lower().isin(judicial_set).astype(int)
    if narrative['is_judicial'].sum() < 2:
        continue
    rho, p_val = spearmanr(narrative['is_judicial'], narrative['ensemble'])
    spearman_rows.append({
        'genre': genre, 'spearman_rho': rho, 'p_value': p_val,
        'n_judicial': int(narrative['is_judicial'].sum()),
        'n_pipeline_narrative': len(narrative),
    })
spearman_df = pd.DataFrame(spearman_rows)
print('\nSpearman ρ per genre:')
print(spearman_df.to_string(index=False))

# Build the validation_table: one row per judicial identification with
# its pipeline lookup result, formatted for paper export.
validation_rows = []
for _, r in elem_matches.iterrows():
    validation_rows.append({
        'cluster_id': r['cluster_id'],
        'case_name': r['caseName'] if 'caseName' in r else '',
        'kind': 'single',
        'judicial_element': r['judicial_element'],
        'genre': r['relevant_genre'],
        'in_pipeline': r.get('in_pipeline', False),
        'pipeline_element': r.get('match_element'),
        'pipeline_element_type': r.get('match_element_type'),
        'pipeline_taxonomy': r.get('element_taxonomy'),
        'ensemble_score': r.get('ensemble_score'),
        'ensemble_rank': r.get('ensemble_rank_in_genre_by_type'),
        'ensemble_percentile': r.get('ensemble_percentile_in_genre_by_type'),
        'flagged_high_stockiness': r.get('flagged_high_stockiness'),
        'match_method': r.get('match_method'),
        'passage_quote': r.get('passage_quote', ''),
    })
val_table = pd.DataFrame(validation_rows)
val_table.to_parquet(PATH_VALIDATION_TBL, index=False)
print(f'\nSaved validation table: {PATH_VALIDATION_TBL} ({len(val_table)} rows)')

# Per-genre PR curve and Spearman go in their own parquets too for paper export
per_genre_pr_df.to_parquet(OUT_DIR / 'precision_recall_per_genre.parquet', index=False)
corpus_pr.to_parquet(OUT_DIR / 'precision_recall_corpus.parquet', index=False)
spearman_df.to_parquet(OUT_DIR / 'spearman_per_genre.parquet', index=False)

In [ ]:
# ----------------------------------------------------------------------
# Phase F — Write the human-readable report
# ----------------------------------------------------------------------

lines = []
def out(line=''):
    print(line); lines.append(line)

out('=' * 70)
out('NOTEBOOK 07: COURT CASE VALIDATION REPORT')
out('=' * 70)
out(f'corpus pull (unique cases)             : {len(unique_cases):,}')
out(f'cases with doctrine passages           : {(passages_df["passage_count"] > 0).sum():,}')
out(f'cases manually triaged                 : {int(triage["manually_verified"].sum())}')
out(f'  scope_tag distribution:')
for tag, n in triage["scope_tag"].value_counts().items():
    out(f'    {tag:<15} {n}')
out(f'judicial single elements extracted     : {int((judicial["kind"]=="single").sum())}')
out(f'judicial combinations extracted        : {judicial[judicial["kind"]=="combination_member"].groupby(["cluster_id","combination_id"]).ngroups}')
out(f'manually-verified judicial rows        : {int(judicial["manually_verified"].sum())}')
out()

out('=' * 70)
out('HEADLINE METRICS')
out('=' * 70)
out()

out('Corpus-wide P@K / R@K / F1@K (mean across genres):')
out(f'  {"K":>4} {"P@K":>8} {"R@K":>8} {"F1@K":>8}')
for _, r in corpus_pr.iterrows():
    out(f'  {int(r["K"]):>4} {r["precision"]:>8.3f} {r["recall"]:>8.3f} {r["f1"]:>8.3f}')
out()
out('Headline lead numbers (per spec):')
out('  Corpus-wide P@25 = ' + f'{corpus_pr.set_index("K").loc[25, "precision"]:.3f}'
    + '   R@25 = ' + f'{corpus_pr.set_index("K").loc[25, "recall"]:.3f}'
    + '   F1@25 = ' + f'{corpus_pr.set_index("K").loc[25, "f1"]:.3f}')

out()
out('Per-genre P@10 (compact table; full curve in precision_recall_per_genre.parquet):')
at_k10 = per_genre_pr_df[per_genre_pr_df['K']==10].sort_values('precision', ascending=False)
out(f'  {"genre":<20} {"P@10":>8} {"R@10":>8} {"F1@10":>8} {"n_judicial":>12}')
for _, r in at_k10.iterrows():
    out(f'  {r["genre"]:<20} {r["precision"]:>8.3f} {r["recall"]:>8.3f} {r["f1"]:>8.3f} {int(r["n_judicial_in_genre"]):>12}')
out()

out('Spearman ρ per genre (directional agreement between judicial labels and pipeline ranking):')
out(f'  {"genre":<20} {"rho":>8} {"p":>10} {"n_jud":>8} {"n_pipe":>8}')
for _, r in spearman_df.iterrows():
    out(f'  {r["genre"]:<20} {r["spearman_rho"]:>8.3f} {r["p_value"]:>10.3e} {int(r["n_judicial"]):>8} {int(r["n_pipeline_narrative"]):>8}')
out()

out('=' * 70)
out('COMBINATION-LEVEL METRICS (top 5% headline, top 1% strict, top 10% generous)')
out('=' * 70)
if len(comb_matches) and 'flagged_top5' in comb_matches.columns:
    matched = comb_matches[comb_matches['pair_in_pipeline']==True]
    out(f'  total judicial pair lookups       : {len(comb_matches)}')
    out(f'  pairs present in pipeline         : {len(matched)}')
    out(f'  precision against top  5% flag    : {matched["flagged_top5"].mean():.3f}')
    out(f'  precision against top  1% flag    : {matched["flagged_top1"].mean():.3f}')
    out(f'  precision against top 10% flag    : {matched["flagged_top10"].mean():.3f}')
else:
    out('  No combination pairs to evaluate.')
out()

out('=' * 70)
out('VALIDATION TABLE (top 30 rows)')
out('=' * 70)
out(val_table.head(30).to_string(index=False, max_colwidth=30))
out()

out('=' * 70)
out('JUDICIAL ELEMENTS WITHOUT A PIPELINE MATCH (judicial-only)')
out('=' * 70)
judicial_only = elem_matches[elem_matches['in_pipeline']==False]
out(f'  count: {len(judicial_only)}  (these are elements judges flagged that the framework does not have in its vocabulary or did not rank)')
for _, r in judicial_only.head(20).iterrows():
    out(f'  {r["judicial_element"]:<40} (genre={r["relevant_genre"]}, hint={r.get("element_type_hint","")})')
if len(judicial_only) > 20:
    out(f'  ... and {len(judicial_only) - 20} more (full list in element_matches.parquet)')

PATH_REPORT.write_text('\n'.join(lines))
print(f'\nReport written to {PATH_REPORT}')

## Outputs produced

Everything writes to `v4_rebuild/07_court_cases/`. The files David needs:

- **`validation_report.txt`** — headline metrics and per-genre tables
- **`validation_table.parquet`** — every judicial identification with its pipeline lookup, the paper's central artifact
- **`element_matches.parquet`** — full per-element match details
- **`combination_matches.parquet`** — full per-pair combination match details
- **`combination_summary.parquet`** — per-combination summaries (min joint count, mean combination_stock, any-flagged booleans)
- **`precision_recall_per_genre.parquet`** — full P@K / R@K / F1@K curves per genre
- **`precision_recall_corpus.parquet`** — corpus-wide aggregates
- **`spearman_per_genre.parquet`** — Spearman ρ per genre
- **`case_triage_CURATED.parquet`** and **`judicial_elements_CURATED.parquet`** — your manually-reviewed ground truth

## Hand-off to David

When the notebook finishes:

1. Verify `validation_report.txt` reads cleanly (open it; check for NaN values; sanity-check the per-genre rows).
2. Confirm the four manual-review files are present and reflect your curated work.
3. **Compress the entire `v4_rebuild/07_court_cases/` folder** and send to David.
4. Send a short note flagging:
   - Cases where the auto-triage or auto-extraction was clearly wrong and you corrected it
   - Cases where the judicial identification is ambiguous and the validation might over- or under-count
   - Any cases where the framework's pipeline match looks surprising (very high or very low rank, or no match where one was expected)

## If anything goes wrong

- **An API call fails or times out**: the per-case caches at `cache_opinions/` and `cache_extraction/` let you resume. Just re-run the cell that failed; cached results are reused.
- **A cell errors midway**: re-run the cell. The checkpoint parquet at the start of each phase means you don't lose prior phases' work.
- **Pipeline output paths are missing**: confirm with David that the v4_rebuild folder structure is in place. He should provide a Drive folder you can mirror into your own Drive.
- **The CourtListener token doesn't work**: confirm it's stored in Colab Secrets as `COURTLISTENER_TOKEN` (capitalization matters) and is enabled for notebook access. Test with a quick `requests.get` to the search endpoint.

When in doubt about a decision, default to the conservative choice (keep the manually_verified flag off, leave a note for David).